In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import matplotlib.pyplot as plt

In [ ]:
def load_data(data_dir, img_size=(128,128)):
    images = []
    masks = []

    #
    for img_file in os.listdir(data_dir):
        if '_mask' not in img_file:
            img_path = os.path.join(data_dir, img_file)
            mask_path = os.path.join(data_dir, img_file.replace('.jpg', '_mask.jpg'))

            #
            img = load_img(img_path, target_size = img_size)
            mask = load_img(mask_path, target_size = img_size, color_mode = 'grayscale')

            #
            img = img_to_array(img) / 255.0
            mask = img_to_array(mask) / 255.0

            images.append(img)
            masks.append(mask)

    return np.array(images), np.array(masks)

In [ ]:
train_dir = '/kaggle/input/bp-dataset2/split_datasets/train'
val_dir = '/kaggle/input/bp-dataset2/split_datasets/val'
test_dir = '/kaggle/input/bp-dataset2/split_datasets/test'

X_train, y_train = load_data(train_dir)
X_val, y_val = load_data(val_dir)
X_test, y_test = load_data(test_dir)

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_val.shape)
print(y_val.shape)
print(X_test.shape)
print(y_test.shape)

In [ ]:
plt.subplot(1, 2, 1)
plt.imshow(X_train[100], cmap='gray')
plt.subplot(1, 2, 2)
plt.imshow(y_train[100], cmap='gray')

In [ ]:
def build_unet_with_group(input_shape=(128, 128, 3), groups=4):
    inputs = layers.Input(input_shape)

    # Expand channels to 32
    x = layers.Conv2D(32, 1, padding='same', activation='relu')(inputs) 
    
    # Encoder
    conv1 = layers.Conv2D(32, 3, groups=groups, activation='relu', padding='same')(x)
    conv1 = layers.Conv2D(32, 3, groups=groups, activation='relu', padding='same')(conv1)
    pool1 = layers.MaxPooling2D(pool_size=(2, 2))(conv1)
    
    conv2 = layers.Conv2D(64, 3, groups=groups, activation='relu', padding='same')(pool1)
    conv2 = layers.Conv2D(64, 3, groups=groups, activation='relu', padding='same')(conv2)
    pool2 = layers.MaxPooling2D(pool_size=(2, 2))(conv2)
    
    conv3 = layers.Conv2D(128, 3, groups=groups, activation='relu', padding='same')(pool2)
    conv3 = layers.Conv2D(128, 3, groups=groups, activation='relu', padding='same')(conv3)
    pool3 = layers.MaxPooling2D(pool_size=(2, 2))(conv3)
    
    # Bridge
    conv4 = layers.Conv2D(256, 3, groups=groups, activation='relu', padding='same')(pool3)
    conv4 = layers.Conv2D(256, 3, groups=groups, activation='relu', padding='same')(conv4)
    
    # Decoder
    up7 = layers.concatenate([layers.Conv2DTranspose(128, 2, strides=(2, 2),padding='same')(conv4), conv3], axis=3)
    conv7 = layers.Conv2D(128, 3, groups=groups, activation='relu', padding='same')(up7)
    conv7 = layers.Conv2D(128, 3, groups=groups, activation='relu', padding='same')(conv7)
    
    up8 = layers.concatenate([layers.Conv2DTranspose(64, 2, strides=(2, 2),padding='same')(conv7), conv2], axis=3)
    conv8 = layers.Conv2D(64, 3, groups=groups, activation='relu', padding='same')(up8)
    conv8 = layers.Conv2D(64, 3, groups=groups, activation='relu', padding='same')(conv8)
    
    up9 = layers.concatenate([layers.Conv2DTranspose(32, 2, strides=(2, 2),padding='same')(conv8), conv1], axis=3)
    conv9 = layers.Conv2D(32, 3, groups=groups, activation='relu', padding='same')(up9)
    conv9 = layers.Conv2D(32, 3, groups=groups, activation='relu', padding='same')(conv9)
    
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(conv9)
    
    model = models.Model(inputs=inputs, outputs=outputs)
    return model


In [ ]:
model = build_unet_with_group()
model.summary()

In [ ]:
def dice_coef(y_true, y_pred, smooth=1e-6):

    y_pred = tf.cast(y_pred > 0.5, dtype=tf.int32)
    y_true = tf.cast(y_true, dtype=tf.float32)
    y_pred = tf.cast(y_pred, dtype=tf.float32)
    
    intersection = tf.reduce_sum(y_true * y_pred)
    intersection = tf.cast(intersection, dtype=tf.float32)

    y_true = tf.reduce_sum(y_true)
    y_pred = tf.reduce_sum(y_pred)
    
    result = (2.0 *(intersection) + smooth) / (y_true + y_pred + smooth)
    
    return result
    
def dice_loss(y_true, y_pred):
    return 1-dice_coef(y_true, y_pred)

def combined_loss(y_true, y_pred, a=0.95):
    logit_loss_value = tf.keras.losses.binary_crossentropy(y_true, y_pred,from_logits=False)
    
    dice_loss_value = dice_loss(y_true, y_pred)
    
    return a*dice_loss_value+ (1-a)*logit_loss_value

In [ ]:
model.compile(optimizer='adam',loss= combined_loss,metrics=[dice_coef])

In [ ]:
# 
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_dice_coef', # validation dice coefficient
    factor=0.9, # learning rate 10%
    patience=3, # 3 epoch   
    min_lr=1e-6, #  learning rate
    mode='max', # dice coefficient 
    verbose=1
)

model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_unet_model.keras', #  
    monitor='val_dice_coef', # validation dice coefficient
    save_best_only=True, #   
    mode='max',
    verbose=1
)

In [ ]:
start = time.time()
# 
history = model.fit(X_train, y_train,
                    batch_size=16,
                    epochs=50,
                    validation_data=(X_val, y_val),
                    callbacks=[reduce_lr, model_checkpoint]
                   )


In [ ]:
#   
best_model = tf.keras.models.load_model('best_unet_model.keras',
                                        custom_objects={
                                            'combined_loss': combined_loss,
                                            'dice_coef': dice_coef})

#
test_loss, test_dice = best_model.evaluate(X_test, y_test)

In [ ]:
#  
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper right')

plt.subplot(1, 2, 2)
plt.plot(history.history['dice_coef'])
plt.plot(history.history['val_dice_coef'])
plt.title('Model dice_coef')
plt.ylabel('dice_coef')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
def visualize_results(model, X, y, num_samples=3):
    
    predictions = model.predict(X[:num_samples])
    
    #
    predictions = (predictions > 0.5).astype(np.float32)
    
    plt.figure(figsize=(4*num_samples, 12))
    for i in range(num_samples):
        plt.subplot(3, num_samples, i + 1)
        plt.imshow(X[i])
        plt.title('Original Image')
        plt.axis('off')
        
        plt.subplot(3, num_samples, i + 1 + num_samples)
        plt.imshow(y[i, :, :, 0], cmap='gray')
        plt.title(f'True Mask\nDice: {dice_coef(y[i:i+1], predictions[i:i+1]).numpy():.4f}')
        plt.axis('off')
        
        plt.subplot(3, num_samples, i + 1 + 2*num_samples)
        plt.imshow(predictions[i, :, :, 0], cmap='gray')
        plt.title('Predicted Mask')
        plt.axis('off')
        
    plt.tight_layout()
    plt.show()

#   
visualize_results(best_model, X_test, y_test)

In [ ]:
#best_model.save('group_model(best2).h5')